In [11]:
from pathlib import Path
import os
import shutil
import subprocess

import pandas as pd
from rdflib import Graph, Namespace, RDF

# 1. Thu muc goc du an
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name.lower() == "notebook":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "ACME_Insurance"
RESULTS_DIR = PROJECT_DIR / "results" / "llm"
assert DATA_DIR.is_dir(), f"Khong tim thay: {DATA_DIR}"

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Environment: OK")

Project: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Data: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\data\ACME_Insurance
Environment: OK


In [12]:
# 2. Kaggle Model Proxy (GPT-5.4 day du chay tren Kaggle, khong chay local)
MODEL = "gpt-5.4-2026-03-05"
KAGGLE_EXE = Path.home() / "kaggle-ai-env" / "Scripts" / "kaggle.exe"
KAGGLE_WORKDIR = Path.home() / "kaggle-gpt54-test"
assert KAGGLE_EXE.is_file(), f"Khong tim thay Kaggle CLI: {KAGGLE_EXE}"
assert KAGGLE_WORKDIR.is_dir(), f"Khong tim thay workspace: {KAGGLE_WORKDIR}"

# Giu UTF-8 khi Kaggle CLI doc prompt DDL/OWL tren Windows
env = os.environ.copy()
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"

# Chi kiem tra ket noi va quota, chua sinh truy van
quota = subprocess.run(
    [str(KAGGLE_EXE), "benchmarks", "quota"],
    cwd=KAGGLE_WORKDIR,
    env=env,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
if quota.returncode != 0:
    raise RuntimeError(quota.stderr or quota.stdout)

print("=== KAGGLE CONNECTION ===")
print("Model:", MODEL)
print(quota.stdout)
print("Status: Ready")

=== KAGGLE CONNECTION ===
Model: gpt-5.4-2026-03-05
period   used   remaining  total    refillAt                    
-------  -----  ---------  -------  --------------------------  
Daily    $3.88  $6.12      $10.00   2026-11-11T01:55:05.842000  
Monthly  $5.19  $94.81     $100.00  2026-12-07T18:49:30.351000  

Status: Ready


In [13]:
# 3. Cau hinh muc tieu theo bai bao goc
TEMPERATURE = 0.3
MAX_TOKENS = 2048
QUERY_TIMEOUT = 60  # Moc tham chieu cua bai bao; Kaggle khong ap dung timeout tung prompt
BATCH_SIZE = 10

print("=== EXPERIMENT CONFIGURATION ===")
print("Model:", MODEL)
print("Temperature:", TEMPERATURE)
print("Max output tokens:", MAX_TOKENS)
print("Requests:", 43 * 2)
print("Batch size:", BATCH_SIZE)
print("Original per-query timeout (not enforced by Kaggle):", QUERY_TIMEOUT, "seconds")

=== EXPERIMENT CONFIGURATION ===
Model: gpt-5.4-2026-03-05
Temperature: 0.3
Max output tokens: 2048
Requests: 86
Batch size: 10
Original per-query timeout (not enforced by Kaggle): 60 seconds


In [14]:
import re

# 1. Đường dẫn DDL và OWL gốc
DDL_FILE = PROJECT_DIR / "data" / "ACME_Insurance" / "DDL" / "ACME_small.ddl"
OWL_FILE = PROJECT_DIR / "data" / "ACME_Insurance" / "ontology" / "insurance.ttl"

# 2. Đọc toàn bộ nội dung, không rút gọn
ddl_text = DDL_FILE.read_text(encoding="utf-8-sig")
owl_text = OWL_FILE.read_text(encoding="utf-8-sig")

# 3. Kiểm tra dữ liệu
assert "CREATE TABLE" in ddl_text.upper()

print("=== ORIGINAL CONTEXT ===")
print("SQL DDL:", len(ddl_text), "characters")
print("OWL Ontology:", len(owl_text), "characters")

=== ORIGINAL CONTEXT ===
SQL DDL: 6154 characters
OWL Ontology: 12563 characters


In [15]:
# 1. Đọc benchmark gốc
BENCHMARK_FILE = DATA_DIR / "investigation" / "acme-benchmark.ttl"

benchmark = Graph()
benchmark.parse(str(BENCHMARK_FILE), format="turtle")

QA = Namespace("http://models.data.world/benchmarks/QandA#")

# 2. Lấy danh sách câu hỏi theo thứ tự URI
questions = []

for inquiry in sorted(benchmark.subjects(RDF.type, QA.Inquiry), key=str):
    questions.append({
        "Inquiry URI": str(inquiry),
        "Question": str(benchmark.value(inquiry, QA.prompt))
    })

# 3. Loại câu hỏi bổ sung ngoài 43 câu của bài báo
extra_question = (
    "Return all the claims by claim number "
    "that were closed in 2019"
)

questions = [
    q for q in questions
    if q["Question"] != extra_question
]

# 4. Tạo bảng câu hỏi và đánh số ID
questions_df = pd.DataFrame(questions)
questions_df.insert(0, "ID", range(1, len(questions_df) + 1))

# 5. Kiểm tra kết quả
assert len(questions_df) == 43
assert questions_df["Question"].nunique() == 43

print("=== BENCHMARK ===")
print("Total questions:", len(questions_df))
display(questions_df.head())


=== BENCHMARK ===
Total questions: 43


,ID,Inquiry URI,Question
0,1,https://templates.data.world/IQ_03fc5a5571db11...,What is the loss ratio of each policy and agen...
1,2,https://templates.data.world/IQ_0d87a2a016062b...,Return all the policies we have by policy numb...
2,3,https://templates.data.world/IQ_1e19282f42eaea...,"What are the loss payment, loss reserve, expen..."
3,4,https://templates.data.world/IQ_1e57233cbc8a71...,"What are the total loss, which is the sum of l..."
4,5,https://templates.data.world/IQ_205ac9d7d19a51...,Return agents and the policy they have sold th...


In [16]:
# 6. Tao task Kaggle voi nhieu prompt doc lap, luu nguyen response
# Khong goi model trong Jupyter local, chi sinh file .py cho Kaggle thuc thi.
def prepare_kaggle_task(task_name, items):
    source = f'''import csv
import time
from pathlib import Path
import kaggle_benchmarks as kbench

ITEMS = {items!r}
TEMPERATURE = {TEMPERATURE}
MAX_TOKENS = {MAX_TOKENS}

@kbench.task(name={task_name!r})
def generate_queries(llm):
    columns = [
        "ID", "Inquiry URI", "Question", "Method",
        "Generated Query", "Prompt Tokens", "Output Tokens",
        "Time (s)", "Status", "Error"
    ]
    with Path("generation.csv").open("w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()

        for item in ITEMS:
            start = time.perf_counter()
            row = {{column: "" for column in columns}}
            for key in ("ID", "Inquiry URI", "Question", "Method"):
                row[key] = item[key]

            try:
                # Mot hoi thoai moi cho moi cau hoi (zero-shot, khong chia se history)
                with kbench.chats.new(f"q-{{item['ID']}}-{{item['Method']}}") as chat:
                    answer = llm.prompt(
                        item["Prompt"],
                        temperature=TEMPERATURE,
                        extra_api_params={{"max_completion_tokens": MAX_TOKENS}},
                    )
                    usage = chat.usage

                row["Generated Query"] = str(answer)
                row["Prompt Tokens"] = getattr(usage, "input_tokens", "")
                row["Output Tokens"] = getattr(usage, "output_tokens", "")
                row["Status"] = "GENERATED" if str(answer).strip() else "EMPTY"
            except Exception as exc:
                row["Status"] = "ERROR"
                row["Error"] = str(exc)

            row["Time (s)"] = round(time.perf_counter() - start, 3)
            writer.writerow(row)
            file.flush()
            print(item["Method"], item["ID"], row["Status"])

generate_queries.run(kbench.llm)
'''
    file_path = KAGGLE_WORKDIR / f"{task_name}.py"
    compile(source, str(file_path), "exec")
    file_path.write_text(source, encoding="utf-8")
    return file_path


def kaggle_cli(*args):
    # Chay Kaggle CLI va hien loi that neu that bai.
    process = subprocess.run(
        [str(KAGGLE_EXE), "benchmarks", "tasks", *map(str, args)],
        cwd=KAGGLE_WORKDIR,
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if process.returncode != 0:
        raise RuntimeError(
            f"kaggle {' '.join(map(str, args))} failed (exit {process.returncode}):\n"
            f"{process.stdout}\n{process.stderr}"
        )
    return process.stdout


print("Kaggle GPT-5.4 generation: Ready")

Kaggle GPT-5.4 generation: Ready


In [17]:
# 7. Giu NGUYEN hai prompt zero-shot cua bai bao
def build_sql_prompt(question):
    return (
        "Given the database described by the following DDL:\n"
        f"{ddl_text}\n"
        "Write a SQL query that answers the following question. "
        "Do not explain the query. return just the query, "
        "so it can be run verbatim from your response.\n"
        "Here’s the question:\n"
        f'"{question}"'
    )


def build_sparql_prompt(question):
    return (
        "Given the OWL model described in the following TTL file:\n"
        f"{owl_text}\n"
        "Write a SPARQL query that answers the question. "
        "The data for your query is available in a SERVICE "
        "identified by <mapped>. Do not explain the query. "
        "return just the query, so it can be run verbatim "
        "from your response.\n"
        "Here’s the question:\n"
        f'"{question}"'
    )

print("Original SQL/SPARQL prompts: Ready")

Original SQL/SPARQL prompts: Ready


In [ ]:
# 8. Cấu hình 50 lần thực nghiệm
RUN_IDS = range(1, 51)

RUN_DIR = RESULTS_DIR / "reproduction"
RUN_DIR.mkdir(parents=True, exist_ok=True)

required_columns = [
    "ID", "Inquiry URI", "Question", "Method",
    "Generated Query", "Prompt Tokens", "Output Tokens",
    "Time (s)", "Status", "Error"
]

# Chuẩn bị 86 câu hỏi SQL/SPARQL (dùng chung cho mọi run)
all_items = []

for _, row in questions_df.iterrows():
    for method, builder in [
        ("SQL", build_sql_prompt),
        ("SPARQL", build_sparql_prompt)
    ]:
        all_items.append({
            "ID": int(row["ID"]),
            "Inquiry URI": row["Inquiry URI"],
            "Question": row["Question"],
            "Method": method,
            "Prompt": builder(row["Question"])
        })

expected_keys = {(x["ID"], x["Method"]) for x in all_items}
assert len(all_items) == len(expected_keys) == 86

# Kiểm tra file kết quả đã hoàn thành
def valid_csv(path, expected):
    if not path.exists():
        return False

    df = pd.read_csv(path, keep_default_na=False)

    return (
        set(required_columns).issubset(df.columns)
        and len(df) == len(expected)
        and set(zip(df["ID"].astype(int), df["Method"])) == expected
        and df["Status"].eq("GENERATED").all()
        and df["Generated Query"].astype(str).str.strip().ne("").all()
    )

print("Runs:", RUN_IDS.start, "to", RUN_IDS.stop - 1)
print("Queries per run:", len(all_items))
print("Total queries:", len(RUN_IDS) * len(all_items))
print("Configuration: OK")

Model: gpt-5.4-2026-03-05
SQL: 43
SPARQL: 43
Total: 86
Batches: 9
Final file: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\llm\reproduction\generation_run_05.csv


In [ ]:
# 9. Thực nghiệm GPT-5.4 từ Run 01 đến Run 50
for run_id in RUN_IDS:
    output_file = RUN_DIR / f"generation_run_{run_id:02d}.csv"

    # Bỏ qua run đã hoàn thành
    if output_file.exists():
        if valid_csv(output_file, expected_keys):
            print(f"Run {run_id:02d}: already completed")
            continue
        raise RuntimeError(f"File chưa hợp lệ: {output_file}")

    batch_dir = RUN_DIR / f"kaggle_batches_run_{run_id:02d}"
    batch_dir.mkdir(parents=True, exist_ok=True)

    batch_files = []

    print(f"\n=== RUN {run_id:02d} ===", flush=True)

    for start in range(0, len(all_items), BATCH_SIZE):
        batch_id = start // BATCH_SIZE + 1
        items = all_items[start:start + BATCH_SIZE]

        task_name = f"acme-gpt54-r{run_id:02d}-b{batch_id:02d}"
        saved = batch_dir / f"{task_name}.csv"
        batch_keys = {(x["ID"], x["Method"]) for x in items}

        # Resume từ CSV đã lưu
        if saved.exists():
            if valid_csv(saved, batch_keys):
                print(f"Batch {batch_id:02d}: reused")
                batch_files.append(saved)
                continue
            raise RuntimeError(f"Batch chưa hợp lệ: {saved}")

        # Tạo task Kaggle
        task_file = prepare_kaggle_task(task_name, items)

        print(f"Batch {batch_id:02d}: push", flush=True)
        kaggle_cli("push", task_name, "-f", task_file, "--wait")

        print(f"Batch {batch_id:02d}: run GPT-5.4", flush=True)
        kaggle_cli("run", task_name, "-m", MODEL, "--wait")

        # Tải kết quả
        download_dir = batch_dir / "downloads" / task_name
        kaggle_cli("download", task_name, "-m", MODEL, "-o", download_dir)

        files = list(download_dir.rglob("generation.csv"))
        if len(files) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất generation.csv: {task_name}")

        df = pd.read_csv(files[0], keep_default_na=False)
        df.to_csv(saved, index=False, encoding="utf-8-sig")

        if not valid_csv(saved, batch_keys):
            raise RuntimeError(f"Kết quả batch chưa hợp lệ: {saved}")

        batch_files.append(saved)
        print(f"Batch {batch_id:02d}: OK", flush=True)

    # Gộp 9 batch thành 1 run
    generation_df = pd.concat(
        [pd.read_csv(f, keep_default_na=False) for f in batch_files],
        ignore_index=True
    )

    generation_df = generation_df.sort_values(
        ["ID", "Method"]
    ).reset_index(drop=True)

    generation_df[required_columns].to_csv(
        output_file, index=False, encoding="utf-8-sig"
    )

    assert valid_csv(output_file, expected_keys)
    print(f"Run {run_id:02d}: COMPLETE (86/86)", flush=True)

print("\nAll requested runs completed.")

[1/9] acme-gpt54-r05-b01: push
[1/9] acme-gpt54-r05-b01: run gpt-5.4-2026-03-05
[1/9] acme-gpt54-r05-b01: download
    Saved 10 rows; status = {'GENERATED': 10}
[2/9] acme-gpt54-r05-b02: push
[2/9] acme-gpt54-r05-b02: run gpt-5.4-2026-03-05
[2/9] acme-gpt54-r05-b02: download
    Saved 10 rows; status = {'GENERATED': 10}
[3/9] acme-gpt54-r05-b03: push
[3/9] acme-gpt54-r05-b03: run gpt-5.4-2026-03-05
[3/9] acme-gpt54-r05-b03: download
    Saved 10 rows; status = {'GENERATED': 10}
[4/9] acme-gpt54-r05-b04: push
[4/9] acme-gpt54-r05-b04: run gpt-5.4-2026-03-05
[4/9] acme-gpt54-r05-b04: download
    Saved 10 rows; status = {'GENERATED': 10}
[5/9] acme-gpt54-r05-b05: push
[5/9] acme-gpt54-r05-b05: run gpt-5.4-2026-03-05
[5/9] acme-gpt54-r05-b05: download
    Saved 10 rows; status = {'GENERATED': 10}
[6/9] acme-gpt54-r05-b06: push
[6/9] acme-gpt54-r05-b06: run gpt-5.4-2026-03-05
[6/9] acme-gpt54-r05-b06: download
    Saved 10 rows; status = {'GENERATED': 10}
[7/9] acme-gpt54-r05-b07: push
[7/

In [ ]:

# 10. Kiểm tra tất cả các run
records = []

for run_id in RUN_IDS:
    file = RUN_DIR / f"generation_run_{run_id:02d}.csv"

    if not file.exists():
        print(f"Run {run_id:02d}: PENDING")
        continue

    assert valid_csv(file, expected_keys), f"Invalid: {file}"

    df = pd.read_csv(file, keep_default_na=False)

    records.append({
        "Run": run_id,
        "SQL": (df["Method"] == "SQL").sum(),
        "SPARQL": (df["Method"] == "SPARQL").sum(),
        "Total": len(df),
        "Markdown fences": df["Generated Query"].str.contains(
            "```", regex=False
        ).sum()
    })

summary_df = pd.DataFrame(records)

print("=== GPT-5.4 GENERATION SUMMARY ===")
print("Completed runs:", len(records), "/", len(RUN_IDS))
print("Total generated queries:", summary_df["Total"].sum())
print("Expected:", len(RUN_IDS) * 86)

display(summary_df)

if len(records) == len(RUN_IDS):
    print("Validation: ALL 50 RUNS COMPLETED")
else:
    print("Validation: INCOMPLETE — có thể tiếp tục Cell 9")

=== GPT-5.4 GENERATION VALIDATION ===
Total: 86
Method  Status   
SPARQL  GENERATED    43
SQL     GENERATED    43
dtype: int64
Queries with Markdown fences: 0
Queries reaching token limit: 0
Queries exceeding original 60s reference: 0
Note: Kaggle does not enforce the original 60s per-query API timeout
Saved: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\llm\reproduction\generation_run_04.csv
Validation: OK
